# EXEMPLO 1 — Técnicas de Query Rewriting
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Demonstrar as 3 técnicas de query rewriting de forma didática e rápida.

**Técnicas cobertas:**
1. Paraphrase Rewriting — reformulação para linguagem técnica
2. HyDE-Lite — documento hipotético como query
3. Step-Back — abstração para nível conceitual

> ⚠️ Este é um exemplo de **referência rápida**. Os labs completos com avaliação comparativa estão em `labs/LAB1_Query_Rewriting.ipynb`.


## ⚙️ 1. Instalação e Configuração

In [ ]:
# Instalar dependências
!pip install langchain langchain-community openai transformers -q
print("✅ Dependências instaladas")

In [ ]:
import os
import json
from typing import List
from dotenv import load_dotenv
from openai import OpenAI

# Carrega ~/mba-rag/.env (variáveis OLLAMA_*, GROQ_*)
load_dotenv()

# ─────────────────────────────────────────────────────────────────
# LLM: Groq primário (llama-3.1-8b-instant) + Ollama fallback
# ─────────────────────────────────────────────────────────────────
GROQ_API_KEY     = os.getenv("GROQ_API_KEY", "")
GROQ_LLM_MODEL   = os.getenv("GROQ_LLM_MODEL", "llama-3.1-8b-instant")
GROQ_BASE_URL    = "https://api.groq.com/openai/v1"
OLLAMA_BASE_URL  = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_LLM_MODEL = os.getenv("OLLAMA_LLM_MODEL", "llama3.2:3b")

client = None
MODEL_NAME = None
LLM_PROVIDER = None

if GROQ_API_KEY:
    try:
        c = OpenAI(base_url=GROQ_BASE_URL, api_key=GROQ_API_KEY)
        c.chat.completions.create(model=GROQ_LLM_MODEL,
                                  messages=[{"role":"user","content":"ok"}],
                                  max_tokens=2, temperature=0.0)
        client, MODEL_NAME, LLM_PROVIDER = c, GROQ_LLM_MODEL, "groq"
    except Exception as e:
        print(f"⚠️  Groq indisponível ({e}); caindo para Ollama…")

if client is None:
    client = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
    MODEL_NAME = OLLAMA_LLM_MODEL
    LLM_PROVIDER = "ollama"

def llm_generate(prompt: str, max_tokens: int = 500) -> str:
    """Wrapper simplificado para chamada ao LLM ativo (Groq ou Ollama)."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

print(f"✅ Cliente LLM configurado | provider={LLM_PROVIDER} | modelo={MODEL_NAME}")

## 📝 2. Query de Exemplo

In [ ]:
# Query original — linguagem coloquial do policial/advogado
query_original = "Podem prender alguém sem mandado?"

print("🔍 Query original:")
print(f"  → {query_original}")
print()
print("❗ Problema: Esta query usa linguagem coloquial.")
print("   Os documentos jurídicos usam termos como:")
print("   'prisão em flagrante', 'art. 302 CPP', 'flagrante delito'")

## 🔄 3. Técnica 1 — Paraphrase Rewriting

In [ ]:
def paraphrase_rewriting(query: str, n: int = 3) -> List[str]:
    """
    Gera N reformulações técnicas da query usando terminologia jurídica.
    
    Args:
        query: Query original em linguagem natural
        n: Número de variações a gerar
    
    Returns:
        Lista de queries reformuladas
    """
    prompt = f"""Você é um especialista jurídico brasileiro com domínio em processo penal.
Reformule a query abaixo em {n} variações usando terminologia técnica jurídica.
Mantenha o mesmo significado, mas use linguagem técnica de documentos jurídicos.

Query original: {query}

Retorne APENAS as {n} variações, uma por linha, sem numeração ou introdução.""""
    
    result = llm_generate(prompt)
    variations = [v.strip() for v in result.split('\n') if v.strip()]
    return variations[:n]

# Executar
print("🔄 TÉCNICA 1: Paraphrase Rewriting")
print("=" * 50)
print(f"Query original: {query_original}")
print()
print("Variações geradas:")
try:
    variations = paraphrase_rewriting(query_original)
    for i, v in enumerate(variations, 1):
        print(f"  {i}. {v}")
except Exception as e:
    # Fallback para demonstração sem LLM
    variations = [
        "Quais são os requisitos para prisão em flagrante delito sem mandado judicial?",
        "Em que hipóteses é admissível a prisão cautelar sem ordem judicial prévia no direito processual penal?",
        "Quais as modalidades de prisão em flagrante previstas no art. 302 do Código de Processo Penal?"
    ]
    print(f"  (Modo demo — LLM não disponível: {e})")
    for i, v in enumerate(variations, 1):
        print(f"  {i}. {v}")

## 🧠 4. Técnica 2 — HyDE-Lite

In [ ]:
def hyde_lite(query: str) -> str:
    """
    Gera um parágrafo hipotético que seria encontrado em documentos jurídicos
    e responderia à query. Usa o embedding desse parágrafo para busca.
    
    ATENÇÃO: Validar sempre o parágrafo gerado — LLMs podem alucinar!
    
    Args:
        query: Query original
    
    Returns:
        Parágrafo hipotético (documento sintético)
    """
    prompt = f"""Você é um redator jurídico especializado em direito processual penal brasileiro.
Escreva um parágrafo curto (4-6 linhas) que seria encontrado em um acórdão, 
código ou doutrina brasileira e que responderia diretamente à pergunta abaixo.

Use linguagem técnica-jurídica formal. Cite artigos de lei ou súmulas quando apropriado.

Pergunta: {query}

Escreva APENAS o parágrafo, sem título ou introdução.""""
    
    return llm_generate(prompt, max_tokens=300)

print("🧠 TÉCNICA 2: HyDE-Lite")
print("=" * 50)
print(f"Query original: {query_original}")
print()
print("Documento hipotético gerado:")
try:
    hyde_doc = hyde_lite(query_original)
    print(f"  [{hyde_doc}]")
except Exception as e:
    hyde_doc = (
        "A prisão sem mandado judicial é admissível nas hipóteses de flagrante delito, "
        "previstas no art. 302 do Código de Processo Penal, que elenca o flagrante próprio, "
        "impróprio, presumido e ficto. Conforme o art. 5º, LXI, da Constituição Federal, "
        "ninguém será preso senão em flagrante delito ou por ordem escrita e fundamentada "
        "de autoridade judiciária competente."
    )
    print(f"  (Modo demo — LLM não disponível)")
    print(f"  [{hyde_doc}]")
print()
print("💡 Este parágrafo será vetorizado e usado como query de busca no lugar da query original.")
print("   O embedding do parágrafo técnico é mais próximo dos documentos do corpus.")

## ↩️ 5. Técnica 3 — Step-Back

In [ ]:
def step_back_query(query: str) -> str:
    """
    Abstrai a query para um nível mais geral, permitindo busca
    em documentos que tratam do conceito antes do detalhe específico.
    
    Args:
        query: Query original
    
    Returns:
        Query abstraída (step-back)
    """
    prompt = f"""Você é um especialista em direito processual penal.
Dada a pergunta específica abaixo, formule uma pergunta MAIS GERAL 
que abrange o conceito jurídico por trás dela.

A pergunta geral deve ser respondida por documentos que contêm a resposta para a específica.

Pergunta específica: {query}

Retorne APENAS a pergunta geral, sem explicação.""""
    
    return llm_generate(prompt, max_tokens=100)

print("↩️ TÉCNICA 3: Step-Back")
print("=" * 50)
print(f"Query original: {query_original}")
print()
try:
    stepback = step_back_query(query_original)
    print(f"Query step-back: {stepback}")
except Exception as e:
    stepback = "Quais são as modalidades de prisão cautelar e seus pressupostos no direito processual penal brasileiro?"
    print(f"  (Modo demo — LLM não disponível)")
    print(f"Query step-back: {stepback}")
print()
print("💡 A busca com a query step-back recupera documentos sobre o tema geral,")
print("   que provavelmente contêm o detalhe específico procurado.")

## 📊 6. Comparação Visual das Técnicas

In [ ]:
# Tabela comparativa das técnicas
import pandas as pd

data = {
    "Técnica": ["Original", "Paraphrase", "HyDE-Lite", "Step-Back"],
    "Query/Texto": [
        query_original,
        variations[0] if variations else "N/A",
        hyde_doc[:80] + "..." if len(hyde_doc) > 80 else hyde_doc,
        stepback
    ],
    "Quando usar": [
        "Baseline",
        "Queries coloquiais simples",
        "Queries técnicas complexas",
        "Queries muito específicas"
    ],
    "Custo LLM": ["0 tokens", "~100 tokens", "~200 tokens", "~50 tokens"]
}

df = pd.DataFrame(data)
print("\n📊 COMPARAÇÃO DAS TÉCNICAS DE QUERY REWRITING")
print("=" * 70)
print(df.to_string(index=False))
print()
print("✅ Exemplo concluído! Veja o LAB1 para comparação com retrieval real.")

## 📚 Referências
- Ma, X. et al. (2023). *Query Rewriting for RAG*. arXiv:2305.14283
- Gao, L. et al. (2022). *HyDE: Precise Zero-Shot Dense Retrieval*. arXiv:2212.10496  
- Zheng, H.S. et al. (2023). *Take a Step Back*. arXiv:2310.06117
